In [1]:

import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding
from tensorflow.keras.models import Model

print("TensorFlow version:", tf.__version__)


TensorFlow version: 2.19.0


In [2]:
input_texts = [
    "hi", "hello", "hey", "good morning", "good afternoon", "good evening",
    "bye", "goodbye", "see you", "take care",
    "thanks", "thank you", "many thanks",
    "yes", "no", "maybe", "of course",
    "sorry", "excuse me", "no problem",
    "please", "welcome", "congratulations",
    "how are you", "how is everything", "how have you been",
    "i am fine", "i am good", "i am okay",
    "what is your name", "my name is", "nice to meet you",
    "where are you from", "i am from india", "i live here",
    "what are you doing", "i am working", "i am studying",
    "do you speak english", "i understand", "i dont understand",
    "can you help me", "help me", "call me",
    "wait here", "come here", "go there",
    "open the door", "close the window", "turn on the light", "turn off the light",
    "i am hungry", "i am tired", "i am happy", "i am sad",
    "lets go", "stop", "start", "run",
    "this is good", "this is bad", "this is amazing",
    "what time is it", "today", "tomorrow", "yesterday",
    "i like this", "i love this", "i hate this",
    "do it now", "do it later", "try again",
    "be careful", "stay safe", "good luck"
]

target_texts = [
    "hola", "hola", "hola", "buenos dias", "buenas tardes", "buenas noches",
    "adios", "adios", "nos vemos", "cuidate",
    "gracias", "gracias", "muchas gracias",
    "si", "no", "quizas", "por supuesto",
    "lo siento", "perdon", "no hay problema",
    "por favor", "bienvenido", "felicidades",
    "como estas", "como esta todo", "como has estado",
    "estoy bien", "estoy bien", "estoy bien",
    "como te llamas", "me llamo", "mucho gusto",
    "de donde eres", "soy de india", "vivo aqui",
    "que estas haciendo", "estoy trabajando", "estoy estudiando",
    "hablas ingles", "entiendo", "no entiendo",
    "puedes ayudarme", "ayudame", "llamame",
    "espera aqui", "ven aqui", "ve alli",
    "abre la puerta", "cierra la ventana", "enciende la luz", "apaga la luz",
    "tengo hambre", "estoy cansado", "estoy feliz", "estoy triste",
    "vamos", "detente", "comienza", "corre",
    "esto es bueno", "esto es malo", "esto es increible",
    "que hora es", "hoy", "manana", "ayer",
    "me gusta esto", "me encanta esto", "odio esto",
    "hazlo ahora", "hazlo despues", "intenta de nuevo",
    "ten cuidado", "mantente seguro", "buena suerte"
]

In [3]:
print("Input texts :", input_texts)
print("Target texts:", target_texts)

Input texts : ['hi', 'hello', 'hey', 'good morning', 'good afternoon', 'good evening', 'bye', 'goodbye', 'see you', 'take care', 'thanks', 'thank you', 'many thanks', 'yes', 'no', 'maybe', 'of course', 'sorry', 'excuse me', 'no problem', 'please', 'welcome', 'congratulations', 'how are you', 'how is everything', 'how have you been', 'i am fine', 'i am good', 'i am okay', 'what is your name', 'my name is', 'nice to meet you', 'where are you from', 'i am from india', 'i live here', 'what are you doing', 'i am working', 'i am studying', 'do you speak english', 'i understand', 'i dont understand', 'can you help me', 'help me', 'call me', 'wait here', 'come here', 'go there', 'open the door', 'close the window', 'turn on the light', 'turn off the light', 'i am hungry', 'i am tired', 'i am happy', 'i am sad', 'lets go', 'stop', 'start', 'run', 'this is good', 'this is bad', 'this is amazing', 'what time is it', 'today', 'tomorrow', 'yesterday', 'i like this', 'i love this', 'i hate this', 'd

In [4]:
target_texts = ["\t" + text + "\n" for text in target_texts]

print("Target texts with start/end symbols:")
for t in target_texts:
    print(repr(t))

Target texts with start/end symbols:
'\thola\n'
'\thola\n'
'\thola\n'
'\tbuenos dias\n'
'\tbuenas tardes\n'
'\tbuenas noches\n'
'\tadios\n'
'\tadios\n'
'\tnos vemos\n'
'\tcuidate\n'
'\tgracias\n'
'\tgracias\n'
'\tmuchas gracias\n'
'\tsi\n'
'\tno\n'
'\tquizas\n'
'\tpor supuesto\n'
'\tlo siento\n'
'\tperdon\n'
'\tno hay problema\n'
'\tpor favor\n'
'\tbienvenido\n'
'\tfelicidades\n'
'\tcomo estas\n'
'\tcomo esta todo\n'
'\tcomo has estado\n'
'\testoy bien\n'
'\testoy bien\n'
'\testoy bien\n'
'\tcomo te llamas\n'
'\tme llamo\n'
'\tmucho gusto\n'
'\tde donde eres\n'
'\tsoy de india\n'
'\tvivo aqui\n'
'\tque estas haciendo\n'
'\testoy trabajando\n'
'\testoy estudiando\n'
'\thablas ingles\n'
'\tentiendo\n'
'\tno entiendo\n'
'\tpuedes ayudarme\n'
'\tayudame\n'
'\tllamame\n'
'\tespera aqui\n'
'\tven aqui\n'
'\tve alli\n'
'\tabre la puerta\n'
'\tcierra la ventana\n'
'\tenciende la luz\n'
'\tapaga la luz\n'
'\ttengo hambre\n'
'\testoy cansado\n'
'\testoy feliz\n'
'\testoy triste\n'
'\tvamos\n'
'\

In [5]:
input_chars = sorted(list(set("".join(input_texts))))
target_chars = sorted(list(set("".join(target_texts))))

input_char_to_index = {char: i + 1 for i, char in enumerate(input_chars)}
target_char_to_index = {char: i + 1 for i, char in enumerate(target_chars)}

input_index_to_char = {i: char for char, i in input_char_to_index.items()}
target_index_to_char = {i: char for char, i in target_char_to_index.items()}

num_encoder_tokens = len(input_chars) + 1
num_decoder_tokens = len(target_chars) + 1

print("Input characters :", input_chars)
print("Target characters:", target_chars)
print("Number of encoder tokens:", num_encoder_tokens)
print("Number of decoder tokens:", num_decoder_tokens)

Input characters : [' ', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'k', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
Target characters: ['\t', '\n', ' ', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'y', 'z']
Number of encoder tokens: 26
Number of decoder tokens: 27


In [6]:
max_encoder_seq_length = max(len(text) for text in input_texts)
max_decoder_seq_length = max(len(text) for text in target_texts)

print("Max encoder sequence length:", max_encoder_seq_length)
print("Max decoder sequence length:", max_decoder_seq_length)

Max encoder sequence length: 20
Max decoder sequence length: 20


In [7]:
encoder_input_data = np.zeros((len(input_texts), max_encoder_seq_length), dtype="int32")
decoder_input_data = np.zeros((len(input_texts), max_decoder_seq_length), dtype="int32")
decoder_target_data = np.zeros((len(input_texts), max_decoder_seq_length, num_decoder_tokens), dtype="float32")

for i, (input_text, target_text) in enumerate(zip(input_texts, target_texts)):
    # Encoder input
    for t, char in enumerate(input_text):
        encoder_input_data[i, t] = input_char_to_index[char]

    # Decoder input and decoder target
    for t, char in enumerate(target_text):
        decoder_input_data[i, t] = target_char_to_index[char]
        if t > 0:
            decoder_target_data[i, t - 1, target_char_to_index[char]] = 1.0

print("Encoder input shape:", encoder_input_data.shape)
print("Decoder input shape:", decoder_input_data.shape)
print("Decoder target shape:", decoder_target_data.shape)

Encoder input shape: (75, 20)
Decoder input shape: (75, 20)
Decoder target shape: (75, 20, 27)


In [8]:
latent_dim = 64

encoder_inputs = Input(shape=(None,), name="encoder_inputs")
encoder_embedding = Embedding(input_dim=num_encoder_tokens, output_dim=16, mask_zero=True, name="encoder_embedding")(encoder_inputs)

encoder_lstm = LSTM(latent_dim, return_state=True, name="encoder_lstm")
encoder_outputs, state_h, state_c = encoder_lstm(encoder_embedding)

encoder_states = [state_h, state_c]

In [9]:
decoder_inputs = Input(shape=(None,), name="decoder_inputs")
decoder_embedding_layer = Embedding(input_dim=num_decoder_tokens, output_dim=16, mask_zero=True, name="decoder_embedding")
decoder_embedding = decoder_embedding_layer(decoder_inputs)

decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True, name="decoder_lstm")
decoder_outputs, _, _ = decoder_lstm(decoder_embedding, initial_state=encoder_states)

decoder_dense = Dense(num_decoder_tokens, activation="softmax", name="decoder_dense")
decoder_outputs = decoder_dense(decoder_outputs)

In [10]:
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_inputs      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_embedding   │ (None, None, 16)  │        416 │ encoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, None)      │          0 │ encoder_inputs[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_embedding   │ (None, None, 16)  │        432 │ decoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm (LSTM) │ [(None, 64),      │     20,736 │ encoder_embeddin… │
│                     │ (None, 64),       │            │ not_equal[0][0]   │
│                     │ (None, 64)]       │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, None,     │     20,736 │ decoder_embeddin… │
│                     │ 64), (None, 64),  │            │ encoder_lstm[0][… │
│                     │ (None, 64)]       │            │ encoder_lstm[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_dense       │ (None, None, 27)  │      1,755 │ decoder_lstm[0][… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 44,075 (172.17 KB)

 Trainable params: 44,075 (172.17 KB)

 Non-trainable params: 0 (0.00 B)

In [11]:
history = model.fit(
    [encoder_input_data, decoder_input_data],
    decoder_target_data,
    batch_size=2,
    epochs=300,
    verbose=1
)

Epoch 1/300
38/38 ━━━━━━━━━━━━━━━━━━━━ 6s 17ms/step - accuracy: 0.1130 - loss: 2.9399
Epoch 2/300
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.1424 - loss: 2.6462
Epoch 3/300
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.1652 - loss: 2.5769
Epoch 4/300
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.1870 - loss: 2.5299
Epoch 5/300
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.1935 - loss: 2.4846
Epoch 6/300
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.2109 - loss: 2.4325
Epoch 7/300
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.2413 - loss: 2.3738
Epoch 8/300
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.2620 - loss: 2.2766
Epoch 9/300
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.2826 - loss: 2.1912
Epoch 10/300
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.2967 - loss: 2.1203
Epoch 11/300
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.3141 - loss: 2.0721
Epoch 12/300
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step

In [12]:
# Encoder inference model
encoder_model = Model(encoder_inputs, encoder_states)

# Decoder inference model
decoder_state_input_h = Input(shape=(latent_dim,), name="decoder_state_input_h")
decoder_state_input_c = Input(shape=(latent_dim,), name="decoder_state_input_c")
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

decoder_embedding_inf = decoder_embedding_layer(decoder_inputs)

decoder_outputs_inf, state_h_inf, state_c_inf = decoder_lstm(
    decoder_embedding_inf, initial_state=decoder_states_inputs
)

decoder_states_inf = [state_h_inf, state_c_inf]
decoder_outputs_inf = decoder_dense(decoder_outputs_inf)

decoder_model = Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs_inf] + decoder_states_inf
)

In [13]:
def decode_sequence(input_seq):
    states_value = encoder_model.predict(input_seq, verbose=0)

    target_seq = np.zeros((1, 1), dtype="int32")
    target_seq[0, 0] = target_char_to_index["\t"]

    decoded_sentence = ""

    while True:
        output_tokens, h, c = decoder_model.predict([target_seq] + states_value, verbose=0)

        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_char = target_index_to_char.get(sampled_token_index, "")

        if sampled_char == "\n" or len(decoded_sentence) > max_decoder_seq_length:
            break

        decoded_sentence += sampled_char

        target_seq = np.zeros((1, 1), dtype="int32")
        target_seq[0, 0] = sampled_token_index

        states_value = [h, c]

    return decoded_sentence

In [14]:
for seq_index in range(len(input_texts)):
    input_seq = encoder_input_data[seq_index: seq_index + 1]
    decoded_sentence = decode_sequence(input_seq)
    print(f"Input: {input_texts[seq_index]}  -->  Predicted Output: {decoded_sentence}")

Input: hi  -->  Predicted Output: hola
Input: hello  -->  Predicted Output: hola
Input: hey  -->  Predicted Output: hola
Input: good morning  -->  Predicted Output: buenos dias
Input: good afternoon  -->  Predicted Output: buenas tardes
Input: good evening  -->  Predicted Output: buenas noches
Input: bye  -->  Predicted Output: adios
Input: goodbye  -->  Predicted Output: adios
Input: see you  -->  Predicted Output: nos vemos
Input: take care  -->  Predicted Output: cuidate
Input: thanks  -->  Predicted Output: gracias
Input: thank you  -->  Predicted Output: gracias
Input: many thanks  -->  Predicted Output: muchas gracias
Input: yes  -->  Predicted Output: si
Input: no  -->  Predicted Output: no
Input: maybe  -->  Predicted Output: quizas
Input: of course  -->  Predicted Output: por supuesto
Input: sorry  -->  Predicted Output: lo siento
Input: excuse me  -->  Predicted Output: perdon
Input: no problem  -->  Predicted Output: no hay problema
Input: please  -->  Predicted Output: por 

In [19]:
def encode_input_text(text):
    seq = np.zeros((1, max_encoder_seq_length), dtype="int32")
    for t, char in enumerate(text[:max_encoder_seq_length]):
        if char in input_char_to_index:
            seq[0, t] = input_char_to_index[char]
    return seq

# Examples
test_words = ["hi", "bye", "yes", "no", "hello", "thanks", "goodbye","good night"]
for word in test_words:
    encoded = encode_input_text(word)
    print(f"Input: {word}  -->  Output: {decode_sequence(encoded)}")

Input: hi  -->  Output: hola
Input: bye  -->  Output: adios
Input: yes  -->  Output: si
Input: no  -->  Output: no
Input: hello  -->  Output: hola
Input: thanks  -->  Output: gracias
Input: goodbye  -->  Output: adios
Input: good night  -->  Output: buenas toy
